# Build Measured Intrinsic Wavefront (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-04-28
**Last Modified:** 2026-04-28
**Status:** Draft
**Keywords:** AOS, intrinsic, double-Zernike, FAM, focal plane

## Description

Build an empirical estimate of the focal-plane intrinsic Zernike map
("measured intrinsic") from FAM donut data, by fitting and subtracting
a chosen sparse subset of Double-Zernike modes (focal `k` × pupil `j`).

Key functionality:
1. Load donut + visit-info parquet pair, filter by day_obs / elevation /
   rotator angle / seq_num.
2. Per visit fit the chosen `(k, j)` DZ modes to `data – tabulated_intrinsic`.
3. Subtract the DZ contribution (only); take the median residual on a
   focal-plane grid per pupil j → iter-1 measured intrinsic.
4. Iterate using iter-1 as the new tabulated intrinsic → iter-2.
5. Compare original / iter-1 / iter-2 / tabulated intrinsic per pupil j
   on a common 5–95 % colour scale.
6. Save iter-2 DZ fits and the measured intrinsic map.

**Output:** parquet (or FITS / HDF5 — see § 8) with `(thx, thy)` in OCS
and CCS plus median `zk` per pupil Noll index, plus a parquet with the
iter-2 per-visit DZ fit coefficients.

**Based on:** `code/measured_intrinsic.py`, `code/dz_fitting.py`,
`code/intrinsics_lib.py`, the `run_pipeline` mktable / fit / plots flow.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-04-28 | Aaron Roodman | Initial version |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [DZ Fit + Iteration](#analysis)
6. [Comparison Plots (4-panel per pupil j)](#plots)
7. [Output Tables](#output)
8. [Output Format Options](#format)

<a id='params'></a>
## 1. Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# ----- Input -----
# Path to the donut parquet file.  The matching visits sidecar
# `{stem}_visits.parquet` is loaded automatically.
donut_parquet = 'output/aos_fam_danish_v1_triplets_20260315_20260409.parquet'

# Coordinate system for the fit and grid: 'OCS' or 'CCS'
coord_sys = 'OCS'

# ----- Filters (None = no constraint) -----
day_obs_min   = None        # e.g. 20260315
day_obs_max   = None        # e.g. 20260409
alt_min_deg   = None        # elevation in degrees
alt_max_deg   = None
rot_min_deg   = None        # rotator_angle in degrees
rot_max_deg   = None
seq_num_min   = None
seq_num_max   = None

# ----- DZ removal spec -----
# Sparse spec: keys are focal Noll k, values are pupil Noll j to fit.
# Default = the user-specified set:
#   k=1 : j = 4..26 except 20, 21      (pentafoil_x/y omitted)
#   k=2 : j = 4..17
#   k=3 : j = 4..17
#   k=4 : j = 4..10
#   k=5 : j = 4..10
#   k=6 : j = 4..10
removal_spec = {
    1: [j for j in range(4, 27) if j not in (20, 21)],
    2: list(range(4, 18)),
    3: list(range(4, 18)),
    4: list(range(4, 11)),
    5: list(range(4, 11)),
    6: list(range(4, 11)),
}

# ----- Iteration -----
n_iter = 2

# ----- Focal-plane grid -----
# The trio plots elsewhere use 18*4 + 1 = 73.  Tunable here.
n_bins = 73
fp_radius_basis = 1.75   # field radius for Z basis normalization (matches OFC)
fp_radius_grid  = 1.8    # grid extent for binned medians

# ----- Output -----
output_dir = None        # None -> derive from donut stem + filter tag
# Format: 'parquet' (recommended), 'hdf5', or 'fits'
output_format = 'parquet'

<a id='setup'></a>
## 2. Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from astropy.table import QTable

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()

sys.path.insert(0, 'code')
from dz_fitting import derive_noll_indices
from measured_intrinsic import (
    expand_removal_spec,
    apply_visit_filters,
    bin_median_focal,
    interpolate_grid_at_donuts,
    build_measured_intrinsic,
    assemble_intrinsic_table,
    save_dz_fits,
)
print('Loaded measured_intrinsic helpers.')

<a id='functions'></a>
## 3. Helper Functions

In [ ]:
def filter_donuts_by_visits(donut_df, visits_kept):
    """Return rows of donut_df whose (day_obs, seq_num) appears in visits_kept."""
    keep_keys = set(zip(
        np.asarray(visits_kept['day_obs']).tolist(),
        np.asarray(visits_kept['seq_num']).tolist(),
    ))
    keys = list(zip(donut_df['day_obs'].tolist(),
                    donut_df['seq_num'].tolist()))
    mask = np.array([k in keep_keys for k in keys])
    return donut_df.loc[mask].reset_index(drop=True)


def common_color_scale(grids, plo=5.0, phi=95.0):
    """Return (vmin, vmax) covering the plo-phi percentile of *all* grids
    (skip NaNs; ignore empty grids)."""
    pooled = np.concatenate(
        [g.ravel()[~np.isnan(g.ravel())] for g in grids if g is not None
         and np.any(np.isfinite(g))])
    if pooled.size == 0:
        return -1.0, 1.0
    return tuple(np.nanpercentile(pooled, [plo, phi]))


def output_dir_default(donut_parquet, coord_sys, day_obs_min, day_obs_max,
                       seq_num_min, seq_num_max,
                       alt_min_deg, alt_max_deg,
                       rot_min_deg, rot_max_deg):
    """Build a self-describing output subdirectory name."""
    stem = Path(donut_parquet).stem
    parts = [stem, f'measured_intrinsic_{coord_sys}']
    if day_obs_min or day_obs_max:
        parts.append(f'day_{day_obs_min or ""}_{day_obs_max or ""}')
    if seq_num_min or seq_num_max:
        parts.append(f'seq_{seq_num_min or ""}_{seq_num_max or ""}')
    if alt_min_deg is not None or alt_max_deg is not None:
        parts.append(f'alt_{alt_min_deg or ""}_{alt_max_deg or ""}')
    if rot_min_deg is not None or rot_max_deg is not None:
        parts.append(f'rot_{rot_min_deg or ""}_{rot_max_deg or ""}')
    return Path('output') / '_'.join(parts)

<a id='data'></a>
## 4. Data Access

In [ ]:
donut_path = Path(donut_parquet)
visits_path = donut_path.parent / f'{donut_path.stem}_visits.parquet'
print(f'Donuts:  {donut_path}')
print(f'Visits:  {visits_path}')
assert donut_path.exists(), f'missing {donut_path}'
assert visits_path.exists(), f'missing {visits_path}'

# Load whole donut table (uses pyarrow under the hood).  For very large
# inputs the streaming variant in measured_intrinsic.fit_dz_modes_streaming
# can be substituted; for now we keep it in-memory because the iterative
# build_measured_intrinsic call needs random access by visit.
donut_full = pd.read_parquet(donut_path)
visits_full = QTable.read(str(visits_path), format='parquet')
print(f'Loaded {len(donut_full)} donuts, {len(visits_full)} visits')

# Apply visit-level filters
visits_kept = apply_visit_filters(
    visits_full,
    day_obs_min=day_obs_min, day_obs_max=day_obs_max,
    alt_min_deg=alt_min_deg, alt_max_deg=alt_max_deg,
    rotator_min_deg=rot_min_deg, rotator_max_deg=rot_max_deg,
    seq_num_min=seq_num_min, seq_num_max=seq_num_max,
)
print(f'Visits after filters: {len(visits_kept)}/{len(visits_full)}')

donut_df = filter_donuts_by_visits(donut_full, visits_kept)
print(f'Donuts after filters: {len(donut_df)}/{len(donut_full)}')

# Derive Noll indices from the data
nZk = np.stack(donut_df[f'zk_{coord_sys}'].values).shape[1]
noll_arr = (np.array(visits_kept['nollIndices'][0])
            if 'nollIndices' in visits_kept.colnames else None)
iZs, iZidx = derive_noll_indices(nZk, noll_arr)
print(f'Pupil Noll indices ({len(iZs)}): {iZs}')

pairs, by_pupil, by_focal = expand_removal_spec(removal_spec)
print(f'\nDZ removal spec: {len(pairs)} (k,j) modes')
for k, js in by_focal.items():
    print(f'  k={k}: j={js[0]}..{js[-1]} ({len(js)} terms)')

<a id='analysis'></a>
## 5. DZ Fit + Iteration

For each visit, fit the chosen `(k, j)` DZ modes to
`data − tabulated_intrinsic`.  Subtract only the DZ contribution from
the data (the intrinsic content is *kept* — it is what we are trying to
measure).  Median over visits → iter-1 measured intrinsic.  Replace the
tabulated intrinsic with the iter-1 grid (linearly interpolated at each
donut's field position) and repeat → iter-2.

The driver `build_measured_intrinsic` performs both iterations and
returns all the intermediate maps + DZ fit tables.

In [ ]:
result = build_measured_intrinsic(
    donut_df, visits_kept, coord_sys, iZs, removal_spec,
    n_iter=n_iter,
    n_bins=n_bins,
    fp_radius_basis=fp_radius_basis,
    fp_radius_grid=fp_radius_grid,
)
print(f'\nIterations completed: {len(result["iter_results"])}')

<a id='plots'></a>
## 6. Comparison Plots — 4-panel per pupil j

For each pupil Noll j (in `iZs`):

| Panel 1 | Panel 2 | Panel 3 | Panel 4 |
|---|---|---|---|
| Original median (raw `zk`) | Iter-1 measured | Iter-2 measured | Tabulated intrinsic |

All four panels share a common colour scale taken from the 5–95 %
percentile of the pooled values across the four panels.

In [ ]:
def plot_4panel_for_j(j, original_grid, iter1_grid, iter2_grid,
                       tabulated_grid, xbins, ybins,
                       coord_sys, plo=5.0, phi=95.0):
    panels = [
        ('Original median (raw zk)', original_grid),
        ('Iter-1 measured', iter1_grid),
        ('Iter-2 measured', iter2_grid),
        ('Tabulated intrinsic', tabulated_grid),
    ]
    grids = [g for _, g in panels]
    vmin, vmax = common_color_scale(grids, plo=plo, phi=phi)

    fig, axes = plt.subplots(1, 4, figsize=(22, 5.5))
    for ax, (title, grid) in zip(axes, panels):
        if grid is None:
            ax.set_visible(False)
            continue
        im = ax.imshow(grid.T, origin='lower',
                       extent=[xbins[0], xbins[-1], ybins[0], ybins[-1]],
                       cmap='viridis', interpolation='none', aspect='equal',
                       vmin=vmin, vmax=vmax)
        plt.colorbar(im, ax=ax, shrink=0.8, label='μm')
        ax.set_xlabel(f'thy_{coord_sys} [deg]')
        ax.set_ylabel(f'thx_{coord_sys} [deg]')
        ax.set_title(title)
    fig.suptitle(
        f'Pupil Z{j}  ({coord_sys})  — color: 5-95% of all 4 panels '
        f'= [{vmin:.3f}, {vmax:.3f}] μm',
        fontsize=13, y=1.02)
    fig.tight_layout()
    return fig


# Build & display the comparison figures
comparison_figs = []
xbins, ybins = result['xbins'], result['ybins']
iter1_grids = result['iter_results'][0]['measured_grid']
iter2_grids = (result['iter_results'][1]['measured_grid']
               if len(result['iter_results']) >= 2
               else result['iter_results'][-1]['measured_grid'])

for j in iZs:
    if j not in by_pupil:
        # not in removal spec; skip (no DZ subtraction defined)
        continue
    fig = plot_4panel_for_j(
        j,
        result['original_median'].get(j),
        iter1_grids.get(j),
        iter2_grids.get(j),
        result['tabulated_median'].get(j),
        xbins, ybins, coord_sys)
    comparison_figs.append(fig)
    plt.show()

print(f'Built {len(comparison_figs)} per-pupil-j comparison figures')

<a id='output'></a>
## 7. Output Tables

In [ ]:
if output_dir is None:
    output_dir = output_dir_default(
        donut_parquet, coord_sys, day_obs_min, day_obs_max,
        seq_num_min, seq_num_max,
        alt_min_deg, alt_max_deg,
        rot_min_deg, rot_max_deg)
output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
print(f'Output directory: {output_dir}')

# 1. Iter-2 (final) DZ fit table — per-visit coefficients keyed dz_z{j}_c{k}
final_iter = result['iter_results'][-1]
dz_fits_path = output_dir / 'measured_intrinsic_dz_fits.parquet'
save_dz_fits(final_iter['fit_rows'], dz_fits_path)

# 2. Measured-intrinsic grid table.  The bin centres live in the chosen
#    coord_sys; the OCS <-> CCS rotation depends on the per-visit
#    camera-rotator angle and therefore is not a single transformation
#    on the binned grid itself.  To get the matching map in the other
#    coordinate system, re-run this notebook with coord_sys='CCS' (or
#    'OCS') — the donut parquet already carries both thx/thy_OCS and
#    thx/thy_CCS plus zk_OCS/zk_CCS.
xcent, ycent = result['xcent'], result['ycent']

intrinsic_table = assemble_intrinsic_table(
    grid=iter2_grids,
    iZs=iZs,
    xcent=xcent, ycent=ycent,
    coord_sys_grid=coord_sys,
    alt_coord_xform=None,
)
print(f'Intrinsic grid table: {len(intrinsic_table)} non-empty bins '
      f'in {coord_sys}')

intrinsic_path = output_dir / f'measured_intrinsic_grid.{output_format}'
if output_format == 'parquet':
    intrinsic_table.write(str(intrinsic_path), format='parquet',
                          overwrite=True)
elif output_format == 'fits':
    intrinsic_table.write(str(intrinsic_path), format='fits',
                          overwrite=True)
elif output_format == 'hdf5':
    intrinsic_table.write(str(intrinsic_path), format='hdf5', path='grid',
                          overwrite=True, append=False)
else:
    raise ValueError(f'Unknown output_format: {output_format}')
print(f'Saved measured intrinsic grid: {intrinsic_path}')

# 3. Composite PDF of all comparison plots
pdf_path = output_dir / 'measured_intrinsic_comparison.pdf'
with PdfPages(pdf_path) as pdf:
    for fig in comparison_figs:
        pdf.savefig(fig, bbox_inches='tight')
print(f'Saved comparison PDF: {pdf_path}')

<a id='format'></a>
## 8. Output Format Options — Rubin DM datasets

The "measured intrinsic" map is a **per-pupil-j focal-plane grid of
median Zernike values** — conceptually parallel to the existing batoid
intrinsic table (`zk_intrinsic_*` columns produced upstream by
`aggregateAOSVisitTable`), but data-derived.  There is **no exact
existing DM dataset type** for "focal-plane gridded measured intrinsic
Zernikes" that I'm aware of.  Realistic choices:

| Option | Pros | Cons |
|---|---|---|
| **Native parquet** under `aos/output/...` (default here) | Matches every other AOS pipeline output; loadable by `astropy.table.QTable.read` and `pandas.read_parquet`; round-trips list columns natively. | Not registered with the Butler — must be passed by path. |
| **FITS BinTable** (`output_format='fits'`) | LSST-DM-friendly, viewable in DS9/topcat, headers carry units. | Slightly heavier than parquet; list columns become variable-length arrays. |
| **HDF5 QTable** (`output_format='hdf5'`) | Multi-table file possible; matches the legacy donut-pair format. | Less standard for AOS outputs going forward. |
| **Custom Butler dataset type** (e.g. `measuredIntrinsicWavefront`) | Full provenance, queryable via `butler.get(...)` once registered, can carry quantum-graph linkage to the FAM data it was built from. | Requires an RFC + StorageClass + dimension records (`instrument`, possibly `physical_filter` / `band`); not a one-line change. |

Suggested path forward:
1. Iterate locally with the **parquet** format (current default) until the
   measurement / iteration approach is settled.
2. Once stable, prototype a Butler dataset type — likely a
   `pandas.DataFrame` or `astropy.QTable` storage class with
   `dimensions = (instrument, band)` (or just `(instrument,)` for a
   filter-averaged map).  At that point the PUT could go through
   `butler.put(intrinsic_table, 'measuredIntrinsicWavefront', ...)`.

The current notebook writes the parquet (and a comparison PDF) to a
self-describing subdirectory under `output/` — both human-readable and
ingestible by a later Butler-registration step.